# 03 — Loading PDFs, Excel, and CSV Files

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Read narrative PDFs (audit memos, policies) with `pypdf` / `pymupdf`.
2. Read multi-sheet Excel workbooks with `pandas.read_excel`.
3. Read transactional CSVs with `pandas.read_csv`.
4. Preview and clean common issues: duplicates, missing PAN, type coercion.


In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


## 3.1 — Read a PDF page-by-page

In [ ]:
from src.document_loaders import load_pdf
pages = load_pdf('data/generated/pdf/02_audit_planning_memo.pdf')
print(f'{len(pages)} page(s)')
print('--- First 500 chars of page 1 ---')
print(pages[0]['text'][:500])
print('--- metadata ---')
print(pages[0]['metadata'])

## 3.2 — Read a multi-sheet Excel

In [ ]:
from src.document_loaders import load_excel
wb = load_excel('data/generated/xlsx/02_general_ledger_sample.xlsx')
print('Sheets:', list(wb))
for name, df in wb.items():
    print(f'\n=== {name} ({len(df)} rows) ===')
    display(df.head(3))

## 3.3 — Read a CSV and quick preview

In [ ]:
import pandas as pd
purchases = pd.read_csv('data/generated/csv/02_purchase_transactions.csv')
print('Shape:', purchases.shape)
print('Columns:', list(purchases.columns))
display(purchases.head())
print('\nApproval status counts:')
print(purchases['approval_status'].value_counts())

## 3.4 — Find the duplicate invoice numbers (teaching trigger)

This is a *real* problem your participants will recognise.

In [ ]:
dupes = purchases[purchases['invoice_no'].duplicated(keep=False)].sort_values('invoice_no')
print(f'Duplicate-invoice rows: {len(dupes)}')
display(dupes[['invoice_no','date','vendor_name','amount','approval_status']])

**Discussion.** Two of the duplicate invoices are under different *vendor names* that look like the same supplier. This is exactly the kind of finding the audit team should flag in the management letter.

## 3.5 — Find missing PAN/VAT entries

In [ ]:
vendors = pd.read_csv('data/generated/csv/04_vendor_master.csv')
missing_pan = vendors[vendors['pan'].isna() | (vendors['pan'] == '')]
display(missing_pan)

## 3.6 — Quick cleaning patterns

In [ ]:
# Normalise vendor names — strip, title-case, collapse spaces
import re
def normalise(name):
    return re.sub(r'\s+', ' ', str(name).strip().title())
vendors['vendor_name_norm'] = vendors['vendor_name'].apply(normalise)
display(vendors[['vendor_code','vendor_name','vendor_name_norm','pan']])

## 3.7 — Cast date columns properly

In [ ]:
purchases['date'] = pd.to_datetime(purchases['date'], errors='coerce')
print('Date range:', purchases['date'].min(), '→', purchases['date'].max())

## Expected output

* Section 3.1 — text of the audit planning memo (page 1).
* Section 3.4 — at least 4 rows with duplicate `invoice_no`.
* Section 3.5 — at least one vendor with empty PAN.


## Exercise

1. Open `data/generated/xlsx/04_accounts_receivable_aging.xlsx` and identify the customer with **all** balances overdue.
2. In the journal-entries CSV, find any JE posted by the same employee who approved it (maker = checker violation).

## Common errors

| Symptom | Fix |
|---|---|
| `FileNotFoundError` | Run `python src/data_generation.py` from the project root |
| `ImportError: openpyxl` | `pip install openpyxl` |
| Garbled PDF text | The PDF is scanned — needs OCR (Notebook 09 covers this limitation) |


## ⚠️ Professional caution

`pandas` is showing you only the rows you ask for. **Always verify the row count (`df.shape`) matches the original source** so you know nothing was silently dropped.